# QLoRA fine-tuning on the ML code mix — Qwen2.5-Coder-0.5B-Instruct

Fine-tunes **`Qwen/Qwen2.5-Coder-0.5B-Instruct`** with QLoRA (4-bit NF4, fp16 compute) on a mix of three ML-code instruction datasets:

| Source | Rows used | License |
|---|---|---|
| `dimsmauls/opencodeinstruct-ml` (ML subset of NVIDIA OpenCodeInstruct) | `OCI_SAMPLES` sampled after the test-score filter | CC BY 4.0 |
| `dimsmauls/magicoder-oss-instruct-ml` (ML subset of Magicoder OSS-Instruct-75K) | all 3,898 | MIT |
| `dimsmauls/code4ml-annotated-rated` (config `training`, split `train`) | all 2,451 | Code4ML (Drozdova et al., 2023) |

LoRA settings follow the thesis (r 8, alpha 16, dropout 0.05, all attention and feed-forward projections, learning rate
2e-4, effective batch 16); the loss is computed on the answer tokens only.

**Colab, free T4:** Runtime → Change runtime type → **T4 GPU**. Add a `HF_TOKEN` secret with **Write** access. Checkpoints
go to Google Drive every `SAVE_STEPS` steps, so when the session ends (free sessions stop after at most ~12 h, often
sooner), reconnect, **Run all** again and training resumes from the last checkpoint. The final adapter is pushed to
`ADAPTER_REPO` (private) with a model card that credits the base model and the datasets.

In [ ]:
%pip install -q -U transformers peft bitsandbytes datasets accelerate huggingface_hub

## 1. Configuration

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
ADAPTER_REPO = 'dimsmauls/qwen2.5-coder-0.5b-ml-mix-qlora'                    # private Hub repo for the final adapter
RUN_NAME = 'qwen2.5-coder-0.5b-ml-mix'
DRIVE_DIR = f'/content/drive/MyDrive/ml-mix-checkpoints/{RUN_NAME}'   # checkpoints survive a disconnect

# Data mix
OCI_SAMPLES = 30_000        # rows sampled from dimsmauls/opencodeinstruct-ml; None = every row left after the filter
OCI_MIN_TEST_SCORE = None   # keep rows with average_test_score >= this; None = no filter. Most ML rows score low in
                            # OpenCodeInstruct (0.9 kept only 18 rows): check the printed distribution before setting one
USE_MAGICODER = True        # all rows of dimsmauls/magicoder-oss-instruct-ml
USE_CODE4ML = True          # the train rows of dimsmauls/code4ml-annotated-rated (config training)
VAL_FRACTION = 0.02         # held out for the validation loss
SEED = 42

# Training (LoRA settings as in the thesis)
MAX_LENGTH = 1024           # tokens per example (prompt + answer); longer examples are cut
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LEARNING_RATE = 2e-4
EPOCHS = 1
BATCH_SIZE, GRAD_ACCUM = 4, 4   # effective batch 16
SAVE_STEPS = 200                # checkpoint (and validation) every 200 optimizer steps

## 2. GPU, Hugging Face login, Google Drive

In [ ]:
import os

os.environ['HF_HUB_DISABLE_XET'] = '1'   # plain HTTP downloads: the Xet backend can crawl at a few hundred kB/s on Colab

import torch
from google.colab import drive, userdata
from huggingface_hub import login

assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('checkpoints in', DRIVE_DIR)

## 3. Data mix

In [ ]:
from collections import Counter

from datasets import concatenate_datasets, load_dataset


def as_pairs(ds, instruction, response, source):
    '''Same three columns for every source: instruction, response, source.'''
    return ds.map(lambda row: {'instruction': row[instruction], 'response': row[response], 'source': source},
                  remove_columns=ds.column_names)


def test_score(row):
    try:
        return float(row['average_test_score'])
    except (TypeError, ValueError):
        return -1.0


parts = []
oci = load_dataset('dimsmauls/opencodeinstruct-ml', split='train')
print('average_test_score in the ML subset (most common):', Counter(oci['average_test_score']).most_common(12))
if OCI_MIN_TEST_SCORE is not None:
    oci = oci.filter(lambda row: test_score(row) >= OCI_MIN_TEST_SCORE)
    print(f'rows with average_test_score >= {OCI_MIN_TEST_SCORE}: {len(oci):,}')
if OCI_SAMPLES and len(oci) > OCI_SAMPLES:
    oci = oci.shuffle(seed=SEED).select(range(OCI_SAMPLES))
parts.append(as_pairs(oci, 'input', 'output', 'opencodeinstruct'))
if USE_MAGICODER:
    parts.append(as_pairs(load_dataset('dimsmauls/magicoder-oss-instruct-ml', split='train'),
                          'problem', 'solution', 'magicoder'))
if USE_CODE4ML:
    code4ml = load_dataset('dimsmauls/code4ml-annotated-rated', 'training', split='train')
    parts.append(as_pairs(code4ml.filter(lambda row: row['split'] == 'train'), 'instruction', 'code', 'code4ml'))

data = concatenate_datasets(parts).shuffle(seed=SEED).train_test_split(test_size=VAL_FRACTION, seed=SEED)
print(data)
print('train rows per source:', dict(Counter(data['train']['source'])))

## 4. Tokenize (chat format, loss on the answer only)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def encode(row):
    '''Chat format of the base model: user = instruction, assistant = response; loss on the response tokens only.'''
    user = [{'role': 'user', 'content': row['instruction']}]
    prompt = tokenizer.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(user + [{'role': 'assistant', 'content': row['response']}], tokenize=False)
    prompt_len = len(tokenizer(prompt, add_special_tokens=False)['input_ids'])
    ids = tokenizer(full, add_special_tokens=False)['input_ids'][:MAX_LENGTH]
    labels = [-100] * min(prompt_len, len(ids)) + ids[prompt_len:]
    return {'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': labels}


tokenized = data.map(encode, remove_columns=data['train'].column_names)
tokenized = tokenized.filter(lambda row: any(label != -100 for label in row['labels']))   # answer cut away entirely
print(tokenized)
lengths = sorted(len(ids) for ids in tokenized['train']['input_ids'])
print('train tokens:', sum(lengths), '| median length:', lengths[len(lengths) // 2], '| max:', lengths[-1])

## 5. 4-bit base model + LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16)   # T4 has no bf16
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant, device_map={'': 0},
                                             torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=TARGET_MODULES, task_type='CAUSAL_LM'))
model.print_trainable_parameters()

## 6. Train (resumes from the last Drive checkpoint)

In [ ]:
import time

from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

args = TrainingArguments(
    output_dir=DRIVE_DIR, num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, lr_scheduler_type='linear',
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    fp16=True, bf16=False, optim='paged_adamw_8bit',
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=20, eval_strategy='steps', eval_steps=SAVE_STEPS, save_strategy='steps', save_steps=SAVE_STEPS,
    save_total_limit=2, seed=SEED, data_seed=SEED, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['test'],
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))

last = get_last_checkpoint(DRIVE_DIR)
print('resuming from', last or 'scratch (no checkpoint yet)')
started = time.time()
result = trainer.train(resume_from_checkpoint=last)
eval_metrics = trainer.evaluate()
train_minutes = round((time.time() - started) / 60, 1)
print(result.metrics, eval_metrics, f'| this session: {train_minutes} min')

## 7. Push the adapter and its model card

In [ ]:
from huggingface_hub import ModelCard

final_dir = f'{DRIVE_DIR}/final'
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
model.push_to_hub(ADAPTER_REPO, private=True)
tokenizer.push_to_hub(ADAPTER_REPO, private=True)

rows = dict(Counter(data['train']['source']))
card = f'''---
base_model: {BASE_MODEL}
library_name: peft
tags:
- qlora
- code
- machine-learning
datasets:
- dimsmauls/opencodeinstruct-ml
- dimsmauls/magicoder-oss-instruct-ml
- dimsmauls/code4ml-annotated-rated
---

# {RUN_NAME} (QLoRA adapter)

QLoRA adapter for **[{BASE_MODEL}](https://huggingface.co/{BASE_MODEL})** (credit and license: the base model's authors
and its model card). Load it on top of the base model with PEFT.

## Training data (credits)

| Dataset | Train rows | Original work |
|---|---|---|
| dimsmauls/opencodeinstruct-ml | {rows.get('opencodeinstruct', 0):,} | ML subset of [OpenCodeInstruct](https://huggingface.co/datasets/nvidia/OpenCodeInstruct) by NVIDIA (Ahmad et al., 2025), CC BY 4.0 |
| dimsmauls/magicoder-oss-instruct-ml | {rows.get('magicoder', 0):,} | ML subset of [Magicoder-OSS-Instruct-75K](https://huggingface.co/datasets/ise-uiuc/Magicoder-OSS-Instruct-75K) (Wei et al., 2023), MIT |
| dimsmauls/code4ml-annotated-rated | {rows.get('code4ml', 0):,} | Built from Code4ML (Drozdova et al., 2023, PeerJ Computer Science) |

OpenCodeInstruct rows: average_test_score >= {OCI_MIN_TEST_SCORE}, {OCI_SAMPLES or 'all'} sampled with seed {SEED}.

## Setup

QLoRA 4-bit NF4 (double quantization, fp16 compute), LoRA r {LORA_R}, alpha {LORA_ALPHA}, dropout {LORA_DROPOUT},
targets {', '.join(TARGET_MODULES)}; learning rate {LEARNING_RATE}, linear schedule, {EPOCHS} epoch(s), effective batch
{BATCH_SIZE * GRAD_ACCUM}, max length {MAX_LENGTH} tokens, loss on the answer tokens only, seed {SEED}.
Final validation loss: {eval_metrics.get('eval_loss', float('nan')):.4f}.
'''
ModelCard(card).push_to_hub(ADAPTER_REPO)
print(f'done: https://huggingface.co/{ADAPTER_REPO}')